In [ ]:
import warnings
warnings.filterwarnings("ignore")
import scanpy as sc
import torch
import numpy as np
import random
import SpaDiff as sd

In [ ]:
parser = sd.get_args() 
args, unknown = parser.parse_known_args() 
Batch_key="omic"

args.n_clusters = 14
args.domains = 14
# args.random_seed = random.randint(1, 1000)
print(args.random_seed)
torch.manual_seed(args.random_seed)
torch.cuda.manual_seed_all(args.random_seed)
np.random.seed(args.random_seed)
random.seed(args.random_seed)
torch.backends.cudnn.deterministic = True

if args.cuda >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{args.cuda}")
    torch.cuda.manual_seed_all(args.random_seed)
    print(f"Using GPU: {torch.cuda.get_device_name(args.cuda)}")
else:
    device = torch.device("cpu")
    print("Using CPU.")

In [ ]:
adata_rna = sc.read_h5ad('E:/gxy_2/2/0_data/case4/SpatialGLUE/Mouse_Brain/adata_RNA.h5ad')
adata_rna.var_names_make_unique()
adata_rna.obs[Batch_key] = "RNA"
adata_atac = sc.read_h5ad('E:/gxy_2/2/0_data/case4/SpatialGLUE/Mouse_Brain/adata_peaks_normalized.h5ad') 
adata_atac.var_names_make_unique()
adata_atac.obs[Batch_key] = "ATAC"

In [ ]:
# sc.pp.filter_cells(adata_rna, min_genes=200)
sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)

adata_rna, HL = sd.spatial_reconstruction(adata_rna,n_neighbors = args.n_neighbors, alpha=args.rec_alpha)
# adata_rna.layers["log1p"] = adata_rna.X.copy()
sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
sc.pp.scale(adata_rna)
adata_rna =  adata_rna[:, adata_rna.var['highly_variable']]

sc.pp.pca(adata_rna, n_comps=50)
X = adata_rna.obsm['X_pca']

sc.pp.highly_variable_genes(adata_atac, flavor="seurat_v3", n_top_genes=3000)
sd.lsi(adata_atac, use_highly_variable=False, n_components=51)
X_lsi = adata_atac.obsm['X_lsi']

In [ ]:
X_rna = torch.from_numpy(X).float().to(device)
X_atac = torch.from_numpy(X_lsi).float().to(device)
HL = {
    k: v.to(device)
    for k, v in HL.items()
}

In [ ]:
model = sd.MultiAutoEncoder(
    rna_dim=X.shape[1],
    atac_dim=X_lsi.shape[1],
    args=args
).to(device)

In [ ]:
loss_hist = sd.train_warmup(model, X_rna, X_atac, HL, args)

dec = sd.DEC(
    n_clusters=args.domains,
    hidden_dim=args.hidden,
    alpha=args.alpha_dec
).to(device)

z, q = dec.fit(
    model=model,
    inputs=(X_rna, X_atac),
    HL=HL,
    n_epochs=args.epochs,
    lr=args.lr,
    lambda_rec=args.recon_weight,
    lambda_attn= 0.05,
    device=device,
    verbose=True
)

labels, z, _ , _ = dec.predict(
    model,
    (X_rna, X_atac),
    HL,
    device=device
)

adata_rna.obsm["z"] = z.numpy()
adata_rna.obs["domain"] = labels.numpy().astype(str)

In [ ]:
adata_rna = sd.adjust_louvain_resolution(adata_rna, args.domains, use_rep='z')
sc.pl.spatial(adata_rna, spot_size=1, color="louvain")